# Transform Sales

This notebook transforms the ETL output by filtering rows and writing to a new table.

In [ ]:
# Widget setup for catalog and schema (safe defaults)
dbutils.widgets.text("catalog", "hive_metastore")
dbutils.widgets.text("schema", "default")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
print(f"➡️ Using catalog={catalog}, schema={schema}")

In [ ]:
import os
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

catalog = os.getenv("DATABRICKS_BUNDLE_VAR_catalog", catalog)
schema = os.getenv("DATABRICKS_BUNDLE_VAR_schema", schema)
if not catalog:
    raise ValueError("❌ No catalog provided. Check databricks.yml target overrides.")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE {schema}")

input_table = f"{catalog}.{schema}.etl_demo_output"
output_table = f"{catalog}.{schema}.sales_transformed"

print(f"Reading input: {input_table}")
df = spark.table(input_table)

df_filtered = df.filter(df.amount_with_tax > 150)

print(f"Writing transformed table: {output_table}")
df_filtered.write.mode("overwrite").saveAsTable(output_table)

print(f"✅ Transformed sales table created at {output_table}")

In [ ]:
display(spark.table(output_table))